In [2]:
import math
import re
from datetime import datetime
import jieba
import pandas as pd
import requests

# 設定 Pandas 顯示寬度，方便在 Jupyter 看完整標題
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.max_columns", 10)
print("✅ Cell 1: 套件與環境初始化完成")

✅ Cell 1: 套件與環境初始化完成


In [3]:
# ==========================================
# 📈 Cell 2：Kalshi 爬蟲模組（過濾遠期合約 + 鎖定近期真實流動性）
# ==========================================


def get_real_kalshi_markets(limit=100, max_days_ahead=365):
  url = "https://api.elections.kalshi.com/trade-api/v2/events"
  params = {"limit": limit, "status": "open", "with_nested_markets": "true"}
  headers = {"accept": "application/json"}

  columns = [
      "ticker",
      "event_title",
      "market_title",
      "category",
      "implied_prob",
      "volume",
      "open_interest",
      "close_date",
  ]

  try:
    resp = requests.get(url, headers=headers, params=params, timeout=12)
    if resp.status_code != 200:
      print(f"❌ Kalshi 請求失敗: {resp.status_code}")
      return pd.DataFrame(columns=columns)

    events = resp.json().get("events", [])
    cleaned = []
    now_utc = datetime.now().astimezone()

    for ev in events:
      ev_title = ev.get("title", "")
      category = ev.get("category", "General")
      markets = ev.get("markets", [])

      # 快速排除明顯是 2028~2040 的超遠期事件標題
      if any(str(yr) in ev_title for yr in range(2028, 2040)):
        continue

      for m in markets:
        # 1. 檢查到期時間，排除超過 1 年以上的超遠期合約
        close_str = m.get("close_time") or m.get("expiration_time")
        close_date_fmt = "N/A"
        if close_str:
          try:
            close_dt = datetime.fromisoformat(close_str.replace("Z", "+00:00"))
            days_diff = (close_dt - now_utc).days
            if days_diff > max_days_ahead or days_diff < 0:
              continue
            close_date_fmt = close_dt.strftime("%Y-%m-%d")
          except Exception:
            pass

        # 2. 取得成交量與未平倉量
        vol = float(
            m.get("volume_fp") or m.get("volume") or m.get("volume_24h") or 0
        )
        oi = float(m.get("open_interest_fp") or m.get("open_interest") or 0)

        # 3. 取得最新報價並轉為機率
        price_raw = (
            m.get("last_price_dollars")
            or m.get("yes_bid_dollars")
            or m.get("yes_ask_dollars")
            or m.get("last_price")
            or m.get("yes_bid")
            or m.get("yes_ask")
        )

        if price_raw is not None:
          try:
            p_val = float(price_raw)
          except ValueError:
            continue
          if p_val <= 0:
            continue
          implied_prob = (
              round(p_val / 100.0, 4) if p_val > 1.0 else round(p_val, 4)
          )
        else:
          continue

        cleaned.append({
            "ticker": m.get("ticker", ""),
            "event_title": ev_title,
            "market_title": (
                m.get("title") or m.get("yes_sub_title") or ev_title
            ),
            "category": category,
            "implied_prob": implied_prob,
            "volume": vol,
            "open_interest": oi,
            "close_date": close_date_fmt,
        })

    df = pd.DataFrame(cleaned, columns=columns)
    if not df.empty:
      df = df.sort_values(by="volume", ascending=False).reset_index(drop=True)
    return df

  except Exception as e:
    print(f"❌ Kalshi 爬蟲異常: {e}")
    return pd.DataFrame(columns=columns)


# 執行並預覽（只抓未來 365 天內結算的近期合約）
df_kalshi = get_real_kalshi_markets(limit=100, max_days_ahead=365)
print(f"✅ Cell 2: 成功抓到 {len(df_kalshi)} 筆【近期】有效 Kalshi 合約")
display(df_kalshi.head(10))

✅ Cell 2: 成功抓到 11 筆【近期】有效 Kalshi 合約


,ticker,event_title,market_title,category,implied_prob,volume,open_interest,close_date
0,KXMLBDEBUT-RCHOLOWSKY-26NOV01,Roch Cholowsky: Debut Date,"Will Roch Cholowsky play in a game for any team in the MLB before Nov 1, 2026?",Sports,0.04,19677.82,11637.84,2026-11-01
1,KXRIEMANNRES-40-27JAN01,When will the Riemann Hypothesis be resolved?,Will the Riemann Hypothesis be resolved before 2027?,Science and Technology,0.10,2811.95,2266.29,2027-01-07
2,KXHODGE-32-27JAN01,When will the Hodge Conjecture be resolved?,Will the Hodge Conjecture be resolved before 2027?,Science and Technology,0.11,2513.72,1789.20,2027-01-07
3,KXNFLRETIRE-KCOUSINS8-2728,Kirk Cousins: Retirement,Will Kirk Cousins announce his retirement before the 2027-28 NFL season?,Sports,0.88,1541.22,774.22,2027-09-30
4,KXFDAAPPROVE-MDMA-27JAN01,Will the FDA approve MDMA for PTSD?,"Will the FDA approve midomafetamine / MDMA for PTSD before Jan 1, 2027?",Science and Technology,0.06,1286.02,671.61,2027-01-01
5,KXBIRCHSWINN-32-27JAN01,When will Birch and Swinnerton-Dyer be resolved?,Will Birch and Swinnerton-Dyer be resolved before 2027?,Science and Technology,0.07,1249.94,980.00,2027-01-07
6,KXNFLRETIRE-MEVANS5-2728,Mike Evans: Retirement,Will Mike Evans announce his retirement before the 2027-28 NFL season?,Sports,0.80,758.64,164.34,2027-09-30
7,KXNFLRETIRE-MSTAFFORD9-2728,Matthew Stafford: Retirement,Will Matthew Stafford announce his retirement before the 2027-28 NFL season?,Sports,0.40,497.83,217.37,2027-09-30
8,KXPNP-32-27JAN01,When will P vs NP be resolved?,Will P vs NP be resolved before 2027?,Science and Technology,0.97,223.66,210.08,2027-01-07
9,KXYANGMILLS-32-27JAN01,When will Yang–Mills and the Mass Gap be resolved?,Will Yang–Mills and the Mass Gap be resolved before 2027?,Science and Technology,0.11,29.69,24.69,2027-01-07


In [4]:
# ==========================================
# 🔍 Cell 1: 掃描並過濾可用模型清單
# ==========================================
import google.generativeai as genai

import os
from dotenv import load_dotenv
load_dotenv()  # 從同資料夾的 .env 讀取金鑰
API_KEY = os.getenv("GEMINI_API_KEY")
genai.configure(api_key=API_KEY)

print("開始掃描此金鑰支援文字生成的可用模型...\n")
try:
    # 篩選出具備 generateContent 能力的模型
    available_models = [m.name for m in genai.list_models() if 'generateContent' in m.supported_generation_methods]
    
    if not available_models:
        print("⚠️ 警告：金鑰有效，但此帳號未開啟任何文字生成模型權限。")
    else:
        print(f"✅ 金鑰驗證通過！共找到 {len(available_models)} 個可用模型。")
        
        # 優先篩選出目前最穩定且無限制的 1.5 系列版本
        print("\n強烈建議使用的穩定版模型 (請複製這些名稱)：")
        recommended = [m for m in available_models if "1.5" in m and ("flash" in m or "pro" in m)]
        for m in recommended:
            print(f"  👉 {m.replace('models/', '')}")
            
        print("\n完整可用清單前 100 筆：")
        for m in available_models[:100]:
            print(f"  - {m}")

except Exception as e:
    print(f"❌ 掃描失敗，網路阻擋或金鑰格式異常：{e}")

開始掃描此金鑰支援文字生成的可用模型...

✅ 金鑰驗證通過！共找到 46 個可用模型。

強烈建議使用的穩定版模型 (請複製這些名稱)：

完整可用清單前 100 筆：
  - models/gemini-2.5-flash
  - models/gemini-2.5-pro
  - models/gemini-2.5-flash-preview-tts
  - models/gemini-2.5-pro-preview-tts
  - models/gemma-4-26b-a4b-it
  - models/gemma-4-31b-it
  - models/gemini-flash-latest
  - models/gemini-flash-lite-latest
  - models/gemini-pro-latest
  - models/gemini-2.5-flash-lite
  - models/gemini-2.5-flash-image
  - models/gemini-3-flash-preview
  - models/gemini-3.1-pro-preview
  - models/gemini-3.1-pro-preview-customtools
  - models/gemini-3.1-flash-lite-preview
  - models/gemini-3.1-flash-lite
  - models/gemini-3-pro-image-preview
  - models/gemini-3-pro-image
  - models/nano-banana-pro-preview
  - models/gemini-3.1-flash-image-preview
  - models/gemini-3.1-flash-image
  - models/gemini-3.1-flash-lite-image
  - models/gemini-nano-banana-2.1
  - models/gemini-3.5-flash
  - models/gemini-3.5-flash-lite
  - models/gemini-omni-flash-preview
  - models/gemini-omni-1

In [5]:
# ==========================================
# 🏥 Cell 2: 全方位 API 狀態與錯誤深度診斷
# ==========================================
import google.generativeai as genai
from google.api_core import exceptions

import os
from dotenv import load_dotenv
load_dotenv()  # 從同資料夾的 .env 讀取金鑰
API_KEY = os.getenv("GEMINI_API_KEY")

# 強制指定一個最普遍支援的 1.5 版模型進行真實通訊測試
# (若 Cell 1 建議其他名稱，可在此替換)
TEST_MODEL = "gemini-3.5-flash" 

genai.configure(api_key=API_KEY)
print(f"啟動深度診斷，測試標的：{TEST_MODEL}\n")

try:
    print("⏳ 正在發送測試封包至 Google API 計費閘道...")
    model = genai.GenerativeModel(TEST_MODEL)
    
    # 發送極短 prompt 以測試實際通訊與權限
    response = model.generate_content("這是一次診斷測試，請回覆『連線正常』四個字。")
    
    print(f"✅ 【診斷通過】API 狀態完美！")
    print(f"   模型實際回覆：{response.text.strip()}")
    print("   結論：金鑰正確、配額充足、未被限速，且模型名稱無誤。程式碼可直接上線抓取新聞！")

except exceptions.InvalidArgument as e:
    print("❌ 【錯誤 400：金鑰格式錯誤】")
    print("   診斷：API Key 字串不完整、多出空白，或包含了不必要的引號。")
    print(f"   系統原錯：{e}")

except exceptions.PermissionDenied as e:
    print("❌ 【錯誤 403：權限遭拒 / 金鑰失效】")
    print("   診斷：這把金鑰已被 Google 停用，或該 GCP 專案的 Generative Language API 服務被關閉。")
    print(f"   系統原錯：{e}")

except exceptions.NotFound as e:
    print("❌ 【錯誤 404：找不到模型】")
    print(f"   診斷：Google 伺服器拒絕了 `{TEST_MODEL}`。這通常是因為你使用的帳號無權存取該特定版本。")
    print("   解法：請回到 Cell 1 挑選清單中有出現的模型名稱替換。")
    print(f"   系統原錯：{e}")

except exceptions.ResourceExhausted as e:
    print("❌ 【錯誤 429：配額耗盡或計費攔截】")
    print("   診斷：你的請求被 Google 計費系統擋下，絕對是以下兩種情況之一：")
    print("   1. [全新專案] 該 GCP 專案未綁定信用卡 (Billing)。目前 Google 經常直接攔截未綁卡專案的首筆請求。")
    print("   2. [已用專案] 你今天的 1,500 次免費呼叫額度 (Quota) 已經徹底歸零。")
    print(f"   系統原錯：{e}")

except Exception as e:
    print(f"❌ 【未知異常：{type(e).__name__}】")
    print("   診斷：非 Google API 限制。可能是本機網路環境被防火牆阻擋，或 requests 模組缺少憑證。")
    print(f"   系統原錯：{e}")

啟動深度診斷，測試標的：gemini-3.5-flash

⏳ 正在發送測試封包至 Google API 計費閘道...
✅ 【診斷通過】API 狀態完美！
   模型實際回覆：連線正常
   結論：金鑰正確、配額充足、未被限速，且模型名稱無誤。程式碼可直接上線抓取新聞！


In [6]:
# ==========================================
# 📰 Cell 3 (終極 NLP 版)：自動化 Jieba 字典 + 暴力爬蟲 + 放寬 AI 逾時
# ==========================================

import google.generativeai as genai
import requests
import pandas as pd
import json
import time
import jieba
import jieba.analyse
from datetime import datetime
from bs4 import BeautifulSoup

# 1. API 初始化
import os
from dotenv import load_dotenv
load_dotenv()  # 從同資料夾的 .env 讀取金鑰
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
genai.configure(api_key=GEMINI_API_KEY)
model = genai.GenerativeModel('gemini-3.5-flash')

# 2. 定義台股白名單
VALID_STOCK_DICT = {
    "2330": "台積電", "2317": "鴻海", "2454": "聯發科", 
    "2308": "台達電", "2881": "富邦金", "3023": "信邦",
    "6672": "騰輝電子", "3376": "新日興", "1503": "士電",
    "1513": "中興電", "1514": "亞力", "1519": "華城",
    "6669": "緯穎", "3406": "玉晶光", "3163": "波若威"
}

# ---------------------------------------------------------
# 💡 核心優化 A：動態注入 Jieba 字典 (財經詞彙 + 台股公司名)
# ---------------------------------------------------------
custom_finance_words = [
    "法說會", "先進製程", "CoWoS", "輝達", "外資", "投信", 
    "自營商", "大盤", "殖利率", "供應鏈", "目標價", "漲停", "跌停"
]
# 加入財經專有名詞
for word in custom_finance_words:
    jieba.add_word(word)
    
# 自動將 DataFrame/Dictionary 裡的所有「台股公司簡稱」加入 Jieba，避免被切碎
for stock_name in VALID_STOCK_DICT.values():
    jieba.add_word(stock_name)

def validate_and_filter_stocks(raw_stocks_list):
    verified_stocks = []
    if not isinstance(raw_stocks_list, list): 
        return verified_stocks
    for item in raw_stocks_list:
        code = str(item).split()[0] if " " in str(item) else str(item)
        if code in VALID_STOCK_DICT:
            verified_stocks.append(f"{code} {VALID_STOCK_DICT[code]}")
    return verified_stocks

# ---------------------------------------------------------
# 💡 核心優化 B：暴力擴大爬蟲範圍 (解決內文過短/爬取失敗)
# ---------------------------------------------------------
def fetch_and_extract_keywords(news_id: str) -> str:
    url = f"https://news.cnyes.com/news/id/{news_id}"
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}
    
    try:
        res = requests.get(url, headers=headers, timeout=8)
        if res.status_code != 200: 
            return ""
        
        soup = BeautifulSoup(res.text, "html.parser")
        
        # 暴力洗淨：強制拔除所有圖片、影片、廣告腳本，只留純文字
        for tag in soup(["script", "style", "img", "video", "iframe", "figure"]):
            tag.decompose()
            
        # 擴展選擇器：針對 VIP、影音版、特殊排版層層遞進，最後用 find_all("p") 兜底
        paragraphs = (soup.select("section.article-body p") or 
                      soup.select("div[itemprop='articleBody'] p") or
                      soup.select("div.news-content p") or
                      soup.find_all("p"))
                      
        raw_text = " ".join([p.text.strip() for p in paragraphs if len(p.text.strip()) > 5])
        
        if len(raw_text) < 50:
            return ""
            
        # 使用剛才吃過大補丸的 Jieba，精準萃取 Top 30 關鍵字
        keywords = jieba.analyse.extract_tags(raw_text, topK=30, withWeight=False)
        return "、".join(keywords)
        
    except Exception as e:
        print(f"    [爬蟲異常] {e}")
        return ""

# ---------------------------------------------------------
# 💡 核心優化 C：放寬 AI 逾時限制 (解決 504 Deadline expired)
# ---------------------------------------------------------
def analyze_sentiment_and_entities(news_title: str, keywords_str: str) -> dict:
    if not keywords_str:
        return {"sentiment_score": 0.0, "affected_stocks": [], "reasoning": "內文過短或純影音，無文字可供分析"}

    prompt = f"""
    任務：台股新聞量化萃取。
    【標題】："{news_title}"
    【內文關鍵字】："{keywords_str}"

    提取規則：
    1. sentiment_score：-1.0(極空) 到 1.0(極多)。中立或大盤請給 0.0。
    2. affected_stocks：只提取受衝擊台股，格式 ["代號 名稱"]。無明確個股則給 []。
    3. reasoning：20字以內客觀事實。
    請只回傳合法 JSON 格式。
    """
    try:
        response = model.generate_content(
            prompt,
            generation_config=genai.types.GenerationConfig(
                temperature=0.0, 
                response_mime_type="application/json"
            ),
            # 將等待時間從 10 秒放寬至 30 秒，容忍 Google 免費伺服器的尖峰延遲
            request_options={"timeout": 30} 
        )
        return json.loads(response.text)
    except Exception as e:
        return {"sentiment_score": 0.0, "affected_stocks": [], "reasoning": f"解析異常: {e}"}

def get_taiwan_news_optimized(limit=10):
    urls = [
        "https://api.cnyes.com/media/api/v1/newslist/category/tw_stock",
        "https://api.cnyes.com/media/api/v1/newslist/category/headline"
    ]
    headers = {"User-Agent": "Mozilla/5.0"}
    columns = ["publish_time", "affected_stocks", "sentiment_score", "reasoning", "title"]
    records = []
    seen_ids = set()

    print(f"⏳ 啟動終極 NLP 降維萃取管線 (預計處理 {limit} 筆)...")
    
    for url in urls:
        try:
            res = requests.get(url, headers=headers, params={"limit": limit}, timeout=8)
            if res.status_code != 200: 
                continue
                
            for it in res.json().get("items", {}).get("data", []):
                if len(records) >= limit: 
                    break
                    
                news_id = it.get("newsId")
                title = it.get("title", "")
                if not news_id or news_id in seen_ids: 
                    continue
                seen_ids.add(news_id)
                
                print(f"  👉 處理進度 [{len(records)+1}/{limit}]: {title[:20]}...")
                pub_time = datetime.fromtimestamp(it.get("publishAt", 0)).strftime("%m-%d %H:%M")
                
                # 執行淨化爬蟲與 Jieba 壓縮
                keywords_str = fetch_and_extract_keywords(news_id)
                
                # 送入 Gemini 評分 (享有 30 秒寬限期)
                llm_result = analyze_sentiment_and_entities(title, keywords_str)
                filtered_stocks = validate_and_filter_stocks(llm_result.get("affected_stocks", []))
                
                records.append({
                    "publish_time": pub_time,
                    "affected_stocks": ", ".join(filtered_stocks),
                    "sentiment_score": llm_result.get("sentiment_score", 0.0),
                    "reasoning": llm_result.get("reasoning", ""),
                    "title": title
                })
                
                time.sleep(2) # 防超速 429 暫停
                
        except Exception as e:
            print(f"❌ 迴圈層級異常: {e}")
            
        if len(records) >= limit: 
            break

    return pd.DataFrame(records, columns=columns)

# ==========================================
# 執行與輸出
# ==========================================
df_news = get_taiwan_news_optimized(limit=10)
print(f"\n✅ Cell 3: 成功提取 {len(df_news)} 筆")
if not df_news.empty:
    display(df_news[["publish_time", "affected_stocks", "sentiment_score", "reasoning", "title"]])

Building prefix dict from the default dictionary ...
Loading model from cache C:\Users\JDA\AppData\Local\Temp\jieba.cache
Loading model cost 0.330 seconds.
Prefix dict has been built successfully.


⏳ 啟動終極 NLP 降維萃取管線 (預計處理 10 筆)...
  👉 處理進度 [1/10]: 國泰世華銀布局企業穩定幣收款與結算服務 ...
  👉 處理進度 [2/10]: 華碩看旺2027年台灣商用PC市場 迎量...
  👉 處理進度 [3/10]: 〈台股盤後〉機器人、記憶體題材熱夯 開高...
  👉 處理進度 [4/10]: OpenAI深化三星合作、日本AI基建升...
  👉 處理進度 [5/10]: 台積擺尾台股守住4萬6、外資連2賣 三大...
  👉 處理進度 [6/10]: 台股連假後小漲54點 旺宏、力成法說聚焦...
  👉 處理進度 [7/10]: 《價值型投資 最新產業研究報告》緯穎 (...
  👉 處理進度 [8/10]: 〈焦點股〉玉晶光攜美商3M推進擴束光學 ...
  👉 處理進度 [9/10]: 〈焦點股〉AMAX-KY 9月獲利1.0...
  👉 處理進度 [10/10]: 〈焦點股〉美系外資調高緯穎目標價 價量齊...

✅ Cell 3: 成功提取 10 筆


,publish_time,affected_stocks,sentiment_score,reasoning,title
0,10-27 15:40,,0.4,國泰世華銀布局企業穩定幣與結算服務。,國泰世華銀布局企業穩定幣收款與結算服務 串聯傳統金融與虛擬資產生態系
1,10-27 15:31,,0.7,華碩看好台灣商用PC市場將迎量價齊揚,華碩看旺2027年台灣商用PC市場 迎量價齊揚與標案預算回溫
2,10-27 15:25,,0.0,"解析異常: 429 You exceeded your current quota, please check your plan and billin...",〈台股盤後〉機器人、記憶體題材熱夯 開高走低小漲54點守住4萬6
3,10-27 15:11,,0.0,"解析異常: 429 You exceeded your current quota, please check your plan and billin...",OpenAI深化三星合作、日本AI基建升溫 野村亞洲雙ETF 00420A 、009833 11/3起限時募集
4,10-27 15:04,,0.0,"解析異常: 429 You exceeded your current quota, please check your plan and billin...",台積擺尾台股守住4萬6、外資連2賣 三大法人買超31.42億元
5,10-27 15:00,,0.0,"解析異常: 429 You exceeded your current quota, please check your plan and billin...",台股連假後小漲54點 旺宏、力成法說聚焦記憶體與AI封裝
6,10-27 14:40,,0.0,"解析異常: 429 You exceeded your current quota, please check your plan and billin...",《價值型投資 最新產業研究報告》緯穎 (6669-TW) Meta與亞馬遜需求延續，液冷機櫃接續成長
7,10-27 14:01,,0.0,"解析異常: 429 You exceeded your current quota, please check your plan and billin...",〈焦點股〉玉晶光攜美商3M推進擴束光學 攻上1245元新天價
8,10-27 13:59,,0.0,"解析異常: 429 You exceeded your current quota, please check your plan and billin...",〈焦點股〉AMAX-KY 9月獲利1.04億元年增51倍 一度攻漲停
9,10-27 13:57,,0.0,"解析異常: 429 You exceeded your current quota, please check your plan and billin...",〈焦點股〉美系外資調高緯穎目標價 價量齊揚漲逾半根停板


In [7]:
# ==========================================
# 📊 Cell 4：即時台股技術面與價量爬蟲 (獨立模組)
# ==========================================
import yfinance as yf
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

def fetch_taiwan_stock_technicals(watchlist):
    """
    抓取自選股的即時價量與技術指標 (RSI, MACD, 布林通道)
    """
    print("⏳ 正在即時爬取台股技術面特徵...")
    tech_records = []
    
    for item in watchlist:
        stock_code = item["code"]
        stock_name = item["name"]
        ticker = f"{stock_code}.TW"
        
        try:
            df = yf.Ticker(ticker).history(period="3mo")
            if df.empty:
                continue
                
            # 計算布林通道
            df['SMA20'] = df['Close'].rolling(window=20).mean()
            df['STD20'] = df['Close'].rolling(window=20).std()
            df['Upper_BB'] = df['SMA20'] + (df['STD20'] * 2)
            df['Lower_BB'] = df['SMA20'] - (df['STD20'] * 2)
            
            # 計算 RSI (14日)
            delta = df['Close'].diff()
            gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
            loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
            df['RSI'] = 100 - (100 / (1 + (gain / loss)))
            
            # 計算 MACD (判斷中期趨勢)
            exp1 = df['Close'].ewm(span=12, adjust=False).mean()
            exp2 = df['Close'].ewm(span=26, adjust=False).mean()
            df['MACD'] = exp1 - exp2
            df['Signal_Line'] = df['MACD'].ewm(span=9, adjust=False).mean()
            df['MACD_Hist'] = df['MACD'] - df['Signal_Line']
            
            latest = df.iloc[-1]
            
            tech_records.append({
                "code": stock_code,
                "name": stock_name,
                "close": latest['Close'],
                "rsi": latest['RSI'],
                "upper_bb": latest['Upper_BB'],
                "lower_bb": latest['Lower_BB'],
                "macd_hist": latest['MACD_Hist'] # 大於0為多頭，小於0為空頭
            })
            print(f"  ✅ {stock_name} ({stock_code}) 技術指標爬取完成")
            
        except Exception as e:
            print(f"  ❌ {stock_name} 爬取失敗: {e}")
            
    return pd.DataFrame(tech_records)

# 執行 Cell 4
watchlist_targets = [
    {"name": "台積電", "code": "2330", "kalshi_keywords": ["semiconductor", "Taiwan", "NVDA", "Nvidia", "Fed", "rate", "TSMC"]},
    {"name": "鴻海", "code": "2317", "kalshi_keywords": ["Apple", "iPhone", "tariff", "supply", "China"]},
    {"name": "聯發科", "code": "2454", "kalshi_keywords": ["semiconductor", "Tech", "Nasdaq", "Inflation"]}
]

df_tech = fetch_taiwan_stock_technicals(watchlist_targets)
display(df_tech)

⏳ 正在即時爬取台股技術面特徵...
  ✅ 台積電 (2330) 技術指標爬取完成
  ✅ 鴻海 (2317) 技術指標爬取完成
  ✅ 聯發科 (2454) 技術指標爬取完成


,code,name,close,rsi,upper_bb,lower_bb,macd_hist
0,2330,台積電,2400.0,15.686275,2608.970454,2344.029546,-16.662011
1,2317,鴻海,247.5,43.689320,257.719119,237.080881,0.190553
2,2454,聯發科,3960.0,6.181818,5403.878146,3713.121854,-95.481147


In [ ]:
# ==========================================
# 🔮 Cell 5：台股五日趨勢預警系統 (結合 AI 與宏觀)
# ==========================================
from datetime import datetime

def generate_5_day_forecast(watchlist, df_n, df_k, df_t):
    """
    根據新聞情緒(短效)、MACD(中效)與宏觀/布林通道(長效)，推算未來 5 天逐日路徑
    """
    print("=" * 70)
    print("🚀 啟動台股五日趨勢預警系統")
    print("=" * 70)
    
    # 1. 計算 Kalshi 總經大盤基底分數 (Cell 2 資料)
    global_macro_score = 0.0
    if not df_k.empty:
        core_keywords = ["Fed", "rate", "inflation", "CPI", "S&P", "Nasdaq", "semiconductor", "Taiwan", "NVDA"]
        combined_titles = df_k["event_title"].fillna("") + " " + df_k["market_title"].fillna("")
        matches = df_k[combined_titles.str.contains("|".join(core_keywords), case=False, na=False)]
        if not matches.empty and matches["volume"].sum() > 0:
            weighted_prob = (matches["implied_prob"] * matches["volume"]).sum() / matches["volume"].sum()
            global_macro_score = weighted_prob - 0.5

    for item in watchlist:
        stock_code = item["code"]
        stock_name = item["name"]
        
        # 2. 獲取 Cell 3 該股的最新 AI 新聞情緒
        sentiment = 0.0
        if not df_n.empty and "affected_stocks" in df_n.columns:
            matched_news = df_n[df_n["affected_stocks"].str.contains(stock_code, case=False, na=False)]
            if not matched_news.empty:
                sentiment = matched_news["sentiment_score"].mean()
                
        # 3. 獲取 Cell 4 該股的技術面
        tech_row = df_t[df_t["code"] == stock_code]
        if tech_row.empty:
            continue
            
        rsi = tech_row.iloc[0]["rsi"]
        macd_hist = tech_row.iloc[0]["macd_hist"]
        close_price = tech_row.iloc[0]["close"]
        upper_bb = tech_row.iloc[0]["upper_bb"]
        lower_bb = tech_row.iloc[0]["lower_bb"]
        
        print(f"\n🎯 【{stock_code} {stock_name}】次週預警報告")
        print(f"   (AI 情緒: {sentiment:+.2f} | RSI: {rsi:.1f} | MACD 動能: {macd_hist:+.2f} | 總經大盤: {global_macro_score:+.2f})")
        print("-" * 50)
        
        # 4. 逐日推演邏輯
        days = ["週一 (T+1)", "週二 (T+2)", "週三 (T+3)", "週四 (T+4)", "週五 (T+5)"]
        for i, day in enumerate(days):
            day_idx = i + 1
            
            # T+1, T+2 受新聞情緒影響最大；T+4, T+5 新聞效應歸零，完全看大盤與均值回歸
            news_weight = max(0, 0.8 - (day_idx * 0.2)) 
            macro_weight = min(0.6, 0.2 + (day_idx * 0.1))
            
            # 當日綜合分數
            daily_score = (sentiment * news_weight) + (global_macro_score * macro_weight)
            
            # 加入技術面干預
            if day_idx <= 3 and macd_hist > 0: 
                daily_score += 0.1 # MACD 偏多，支撐前三日
            elif day_idx <= 3 and macd_hist < 0:
                daily_score -= 0.1 # MACD 偏空，壓抑前三日
                
            # 極端值反轉預測 (到了週四、週五，極易觸碰布林通道而均值回歸)
            if rsi >= 75 or close_price >= upper_bb:
                if day_idx >= 3: daily_score -= 0.3 # 預計週三後漲多回檔
            elif rsi <= 25 or close_price <= lower_bb:
                if day_idx >= 3: daily_score += 0.3 # 預計週三後跌深反彈
                
            # 判定燈號
            if daily_score >= 0.15:
                trend = "🔴 收漲 (機率偏高)"
            elif daily_score <= -0.15:
                trend = "🟢 收跌 (機率偏高)"
            else:
                trend = "⚪ 震盪 / 盤整"
                
            print(f"   🗓️ {day} : {trend}  (推算動能: {daily_score:+.2f})")

# 執行 Cell 5 (請確認 df_news 與 df_kalshi 在記憶體中存在)
if 'df_news' not in locals(): df_news = pd.DataFrame()
if 'df_kalshi' not in locals(): df_kalshi = pd.DataFrame()

generate_5_day_forecast(watchlist_targets, df_news, df_kalshi, df_tech)